# nb_cdf_incremental_pattern — CDF-driven deltas without MLV, watermarked in SQL Database
**The pattern:** Change Data Feed supplies row-level changes; a **watermark table in Fabric SQL
Database** supplies the resumption point. Together they give you incremental silver→gold processing
with none of the MLV constraints (MLV needs Spark-SQL definitions and Delta sources, and manages
refresh on its own schedule) — you keep full control of batching, ordering and error handling.

**What this notebook proves, executed:**
1. CDF version bookkeeping in a SQL metadata table (SQLite locally, Fabric SQL DB by parameter).
2. Reading changes with **version bounds pushed into the source** (`startingVersion` / `endingVersion`)
   rather than reading everything and filtering afterwards.
3. Predicate + column pushdown *on top of* the version bounds, verified in the physical plan.
4. Net-change collapsing so a MERGE applies one row per key, not every intermediate state.
5. **Idempotency** — a second run processes zero rows.
6. The **retention trap** — detecting that a stored watermark has aged out of CDF retention and
   failing over to full refresh instead of silently losing data.

In [1]:
NOTEBOOK_NAME  = "nb_cdf_incremental_pattern"
TABLES_ROOT    = "/tmp/fabric_cdf_demo"
META_TARGET    = "local"          # "local" (SQLite) | "fabric" (Fabric SQL Database)
META_DB        = "/tmp/etl_cdf_meta.db"
ENTITY_NAME    = "silver_orders__gold_daily"
MAX_VERSIONS_PER_RUN = 50         # bound the batch so catch-up never becomes one huge job

In [2]:
import os, json, shutil, sqlite3
from datetime import datetime, timezone
from pyspark.sql import SparkSession, functions as F

# --- Session: Fabric is the default target -------------------------------------
# In Fabric you do NOT create a Spark session. The Livy layer starts it before your first
# cell runs, and `spark` (plus `sc`, `notebookutils`) are already bound. Calling
# SparkSession.builder there is at best a no-op via getOrCreate() and at worst misleading:
# master(), Delta wiring and executor shape are all decided by the Environment/pool, not here.
#
# Session-start settings belong in a %%configure -f cell ABOVE this one, or in the
# Environment's Spark properties. Only runtime-mutable keys can be set from code.
try:
    spark                      # noqa: F821  <- Fabric (and any live session): already provided
    IN_FABRIC = True
except NameError:
    # Local/dev fallback ONLY. Never runs in Fabric.
    IN_FABRIC = False
    from pyspark.sql import SparkSession
    from delta import configure_spark_with_delta_pip
    _b = (SparkSession.builder.appName(NOTEBOOK_NAME).master("local[4]")
          .config("spark.driver.memory", "2g")
          .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
          .config("spark.sql.catalog.spark_catalog",
                  "org.apache.spark.sql.delta.catalog.DeltaCatalog"))
    spark = configure_spark_with_delta_pip(_b).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print(("Fabric session (provided)" if IN_FABRIC else "local session (dev fallback)"),
      "| Spark", spark.version)

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
SILVER = f"{TABLES_ROOT}/silver_orders"
GOLD   = f"{TABLES_ROOT}/gold_daily"
print("run", RUN_ID)


26/08/14 13:18:22 WARN Utils: Your hostname, vm resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/08/14 13:18:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/usr/local/lib/python3.12/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-aae2fd72-f36e-48d6-8c36-34dadb4e3232;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central


	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 239ms :: artifacts dl 14ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0   ||   3   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-aae2fd72-f36e-48d6-8c36-34dadb4e3232
	confs: [default]
	0 artifacts copied, 3 already retrieved (0kB/3ms)


26/08/14 13:18:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


local session (dev fallback) | Spark 3.5.1
run 20260814T131826Z


## 1 — Watermark table (Fabric SQL Database in production)

In [3]:
def meta_connect():
    if META_TARGET == "fabric":
        # Same pyodbc + Entra pattern as nb_metadata_sqldb_prototype (see Sec 27 gotchas).
        import pyodbc, struct, notebookutils
        token = notebookutils.credentials.getToken("https://database.windows.net/").encode("utf-16-le")
        tok = struct.pack(f"<I{len(token)}s", len(token), token)
        drv = sorted(d for d in pyodbc.drivers() if "ODBC Driver" in d)[-1]
        return pyodbc.connect(f"Driver={{{drv}}};Server={SQL_SERVER},1433;Database={SQL_DATABASE};"
                              f"Encrypt=yes;", attrs_before={1256: tok})
    return sqlite3.connect(META_DB)

DDL_TSQL = """
-- Fabric SQL Database
CREATE TABLE dbo.etl_cdf_watermark (
  entity_name       NVARCHAR(200) NOT NULL PRIMARY KEY,
  source_table      NVARCHAR(400) NOT NULL,
  last_version      BIGINT        NOT NULL,   -- last Delta version SUCCESSFULLY processed
  last_run_id       NVARCHAR(40)  NULL,
  rows_applied      BIGINT        NULL,
  updated_at        DATETIME2     NOT NULL DEFAULT SYSUTCDATETIME()
);
"""
if META_TARGET == "local" and os.path.exists(META_DB): os.remove(META_DB)
mcon = meta_connect(); mcur = mcon.cursor()
mcur.execute("""CREATE TABLE IF NOT EXISTS etl_cdf_watermark(
  entity_name TEXT PRIMARY KEY, source_table TEXT NOT NULL, last_version INTEGER NOT NULL,
  last_run_id TEXT, rows_applied INTEGER, updated_at TEXT NOT NULL)""")
mcon.commit()
print("watermark table ready |", DDL_TSQL.strip().splitlines()[1])

watermark table ready | CREATE TABLE dbo.etl_cdf_watermark (


## 2 — Source table with CDF enabled, plus some change history

In [4]:
shutil.rmtree(TABLES_ROOT, ignore_errors=True)
spark.sql(f"""CREATE TABLE IF NOT EXISTS delta.`{SILVER}`
  (order_id BIGINT, customer_id INT, amount DOUBLE, status STRING, order_date DATE)
  USING DELTA
  TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true',
                 'delta.enableDeletionVectors' = 'true')""")

# v1: initial load
(spark.range(0, 20000)
   .withColumn("order_id", F.col("id"))
   .withColumn("customer_id", (F.col("id") % 500).cast("int"))
   .withColumn("amount", F.round(F.rand(7)*200 + 10, 2))
   .withColumn("status", F.lit("complete"))
   .withColumn("order_date", F.date_add(F.lit("2026-07-01"), (F.col("id") % 10).cast("int")))
   .select("order_id","customer_id","amount","status","order_date")
   .write.format("delta").mode("append").save(SILVER))

hist = lambda: spark.sql(f"DESCRIBE HISTORY delta.`{SILVER}`").select("version","operation").collect()
print("versions after initial load:", [(h["version"], h["operation"]) for h in hist()])

versions after initial load: [(1, 'WRITE'), (0, 'CREATE TABLE')]


## 3 — Establish the baseline watermark, then build gold once

In [5]:
def current_version(path):
    return spark.sql(f"DESCRIBE HISTORY delta.`{path}` LIMIT 1").collect()[0]["version"]

def get_watermark(entity):
    r = mcur.execute("SELECT last_version FROM etl_cdf_watermark WHERE entity_name = ?", (entity,)).fetchone()
    return r[0] if r else None

def set_watermark(entity, source, version, rows):
    # Forward-only upsert - concurrent runs can never move the watermark backwards.
    mcur.execute("""INSERT INTO etl_cdf_watermark(entity_name, source_table, last_version,
                       last_run_id, rows_applied, updated_at) VALUES (?,?,?,?,?,?)
                    ON CONFLICT(entity_name) DO UPDATE SET
                       last_version = excluded.last_version, last_run_id = excluded.last_run_id,
                       rows_applied = excluded.rows_applied, updated_at = excluded.updated_at
                    WHERE etl_cdf_watermark.last_version < excluded.last_version""",
                 (entity, source, version, RUN_ID, rows, datetime.now(timezone.utc).isoformat()))
    mcon.commit()

# Seed gold from a full read once, and record the version it reflects.
v0 = current_version(SILVER)
(spark.read.format("delta").load(SILVER)
   .where("status = 'complete'")
   .groupBy("order_date").agg(F.sum("amount").alias("revenue"), F.count("*").alias("orders"))
   .write.format("delta").mode("overwrite").save(GOLD))
set_watermark(ENTITY_NAME, SILVER, v0, 20000)
print(f"gold seeded from version {v0} | watermark = {get_watermark(ENTITY_NAME)}")
spark.read.format("delta").load(GOLD).orderBy("order_date").show(3)

gold seeded from version 1 | watermark = 1


+----------+------------------+------+
|order_date|           revenue|orders|
+----------+------------------+------+
|2026-07-01|217799.89000000007|  2000|
|2026-07-02| 218212.0200000001|  2000|
|2026-07-03|217731.00000000003|  2000|
+----------+------------------+------+
only showing top 3 rows



## 4 — Generate real change traffic: inserts, updates, deletes

In [6]:
from delta.tables import DeltaTable
# inserts (append -> no CDC files written; derived from added files)
(spark.range(20000, 23000)
   .withColumn("order_id", F.col("id"))
   .withColumn("customer_id", (F.col("id") % 500).cast("int"))
   .withColumn("amount", F.round(F.rand(3)*200 + 10, 2))
   .withColumn("status", F.lit("complete"))
   .withColumn("order_date", F.date_add(F.lit("2026-07-01"), (F.col("id") % 10).cast("int")))
   .select("order_id","customer_id","amount","status","order_date")
   .write.format("delta").mode("append").save(SILVER))
# updates (row-changing -> CDC files WITH preimage/postimage)
spark.sql(f"UPDATE delta.`{SILVER}` SET amount = amount * 1.10 WHERE order_id < 500")
# deletes (row-changing -> CDC delete records, even with deletion vectors on)
spark.sql(f"DELETE FROM delta.`{SILVER}` WHERE order_id BETWEEN 1000 AND 1200")
# maintenance (NOT data-changing -> emits NO CDF rows; proven below)
spark.sql(f"OPTIMIZE delta.`{SILVER}`")
print("history:", [(h["version"], h["operation"]) for h in hist()][:6])
print("cdc files present:", os.path.isdir(f"{SILVER}/_change_data"))

history: [(5, 'OPTIMIZE'), (4, 'DELETE'), (3, 'UPDATE'), (2, 'WRITE'), (1, 'WRITE'), (0, 'CREATE TABLE')]
cdc files present: True


## 5 — Read the delta, with the bounds pushed into the source
`startingVersion` / `endingVersion` are **source options, not filters** — Delta uses them to decide
which commits and CDC files to open. That is the real pushdown here: files that fall outside the
version window are never read. Column projection and predicates then push down *within* those files.

In [7]:
def read_changes(path, from_version, to_version, select_cols, predicate=None):
    df = (spark.read.format("delta")
            .option("readChangeFeed", "true")
            .option("startingVersion", from_version + 1)   # exclusive of what we already applied
            .option("endingVersion",  to_version)          # bounded batch - no unbounded catch-up
            .load(path))
    # 1) drop preimages immediately: they are the OLD values, never applied forward
    df = df.where(F.col("_change_type") != "update_preimage")
    # 2) business predicate - pushes into the CDC/data file scan
    if predicate:
        df = df.where(predicate)
    # 3) column projection - keeps ReadSchema narrow
    return df.select(*select_cols, "_change_type", "_commit_version")

wm = get_watermark(ENTITY_NAME)
latest = current_version(SILVER)
target = min(latest, wm + MAX_VERSIONS_PER_RUN)
print(f"watermark={wm} latest={latest} -> processing versions {wm+1}..{target}")

changes = read_changes(SILVER, wm, target,
                       ["order_id","order_date","amount","status"],
                       "status = 'complete'")
changes.groupBy("_change_type").count().show()

watermark=1 latest=5 -> processing versions 2..5


+----------------+-----+
|    _change_type|count|
+----------------+-----+
|update_postimage|  500|
|          insert| 3000|
|          delete|  201|
+----------------+-----+



In [8]:
# Verify the pushdown actually happened rather than trusting the API.
plan = changes._jdf.queryExecution().explainString(
    changes._sc._jvm.org.apache.spark.sql.execution.ExplainMode.fromString("formatted"))
import re
pushed = re.findall(r"PushedFilters: \[([^\]]*)\]", plan)
schema = re.findall(r"ReadSchema: struct<([^>]*)>", plan)
print("PushedFilters:", pushed[:2])
print("ReadSchema cols:", [len(s.split(",")) for s in schema[:2]], "(narrow = pruning reached the scan)")
assert any("status" in p for p in pushed), "business predicate did not push down"
print("PASS: version window bounds the files read; predicate and projection push into the scan")

PushedFilters: ['IsNotNull(_change_type), IsNotNull(status), Not(EqualTo(_change_type,update_preimage)), EqualTo(status,complete)']
ReadSchema cols: [6] (narrow = pruning reached the scan)
PASS: version window bounds the files read; predicate and projection push into the scan


## 6 — Collapse to net change per key, then MERGE once
CDF can contain several rows for the same key across versions (inserted, then updated twice).
Applying them one by one is both slow and wrong-order-sensitive; take the **last state per key by
`_commit_version`** and apply a single row.

In [9]:
from pyspark.sql.window import Window

w = Window.partitionBy("order_id").orderBy(F.col("_commit_version").desc())
net = (changes.withColumn("rn", F.row_number().over(w)).where("rn = 1").drop("rn"))
print("raw change rows:", changes.count(), "-> net rows per key:", net.count())

# Only the dates touched by this batch need recomputing - the second pushdown that matters.
touched = [r["order_date"] for r in net.select("order_date").distinct().collect()]
print(f"affected partitions/dates: {len(touched)} (recompute only these, not the whole table)")

recomputed = (spark.read.format("delta").load(SILVER)
    .where(F.col("order_date").isin(touched) & (F.col("status") == "complete"))  # pushes to scan
    .groupBy("order_date").agg(F.sum("amount").alias("revenue"), F.count("*").alias("orders")))

(DeltaTable.forPath(spark, GOLD).alias("t")
   .merge(recomputed.alias("s"), "t.order_date = s.order_date")
   .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())

set_watermark(ENTITY_NAME, SILVER, target, net.count())
print("gold updated | watermark ->", get_watermark(ENTITY_NAME))
spark.read.format("delta").load(GOLD).orderBy("order_date").show(3)

raw change rows: 3701 -> net rows per key: 3701


affected partitions/dates: 10 (recompute only these, not the whole table)


gold updated | watermark -> 5


+----------+------------------+------+
|order_date|           revenue|orders|
+----------+------------------+------+
|2026-07-01|249419.62700000036|  2279|
|2026-07-02|248769.84400000013|  2280|
|2026-07-03|249235.64500000014|  2280|
+----------+------------------+------+
only showing top 3 rows



## 7 — Idempotency, and proof that OPTIMIZE emitted no change rows

In [10]:
wm2 = get_watermark(ENTITY_NAME)
latest2 = current_version(SILVER)
print(f"watermark={wm2} latest={latest2}")
if wm2 >= latest2:
    print("nothing new - run is a no-op (correct)")
    rows2 = 0
else:
    # Versions after the watermark exist only because OPTIMIZE ran; it is not data-changing,
    # so the change feed for that window must be empty.
    c2 = read_changes(SILVER, wm2, latest2, ["order_id","order_date","amount","status"])
    rows2 = c2.count()
    print(f"versions {wm2+1}..{latest2} contain {rows2} change rows "
          f"(operations: {[h['operation'] for h in hist() if h['version'] > wm2]})")
assert rows2 == 0, "second run found work to do - idempotency broken"
print("PASS: re-running processes zero rows; maintenance operations produce no CDF churn")

watermark=5 latest=5
nothing new - run is a no-op (correct)
PASS: re-running processes zero rows; maintenance operations produce no CDF churn


## 8 — The retention trap, handled
CDF data lives under `_change_data/` and is subject to VACUUM retention (default 7 days); the log
versions themselves age out too. If a consumer stalls longer than retention, its stored watermark
becomes unreadable — the failure you want is a **loud fallback to full refresh**, not a silent gap.

## 9 — Bonus: diagnosing why an MLV would fall back to full refresh
If `silver_orders` above were an MLV source instead of feeding a hand-rolled MERGE, would Fabric's
Optimal Refresh go incremental? Five conditions gate it (Sec 20 of the internals doc); this checks
each one against a real table using only what's already open in this session.

In [11]:
def diagnose_mlv_refresh(table_path: str, since_version: int = None):
    """Checks the conditions that force an MLV referencing this table to full-refresh.
    Mirrors the diagnostic in Sec 20 of the internals doc - not an official Fabric API,
    since MLV refresh-strategy metadata isn't queryable this way; this infers from the
    same signals a human would check manually."""
    props = {r["key"]: r["value"] for r in spark.sql(f"SHOW TBLPROPERTIES delta.`{table_path}`").collect()}
    cdf_on = props.get("delta.enableChangeDataFeed") == "true"
    hist = spark.sql(f"DESCRIBE HISTORY delta.`{table_path}`").select("version", "operation").collect()
    since = since_version if since_version is not None else max(0, hist[0]["version"] - 5)
    recent_ops = {h["operation"] for h in hist if h["version"] > since}
    has_mutation = bool(recent_ops & {"UPDATE", "DELETE", "MERGE"})

    verdict = "INCREMENTAL-ELIGIBLE" if (cdf_on and not has_mutation) else "WOULD FULL-REFRESH"
    reasons = []
    if not cdf_on:
        reasons.append("CDF not enabled on this source - required on EVERY source table")
    if has_mutation:
        reasons.append(f"non-append operation(s) since v{since}: {sorted(recent_ops & {'UPDATE','DELETE','MERGE'})} "
                       f"- incremental refresh requires append-only source data, even with CDF on")
    return {"table": table_path, "cdf_enabled": cdf_on, "recent_operations": sorted(recent_ops),
            "verdict": verdict, "reasons": reasons or ["append-only since checkpoint, CDF on - no blocker found"]}

report = diagnose_mlv_refresh(SILVER)
print(json.dumps(report, indent=2))
if report["verdict"] == "WOULD FULL-REFRESH":
    print("\n-> This is exactly the case nb_cdf_incremental_pattern (this notebook) exists to handle:")
    print("   CDF + a SQL Database watermark, doing what MLV's append-only requirement excludes.")

{
  "table": "/tmp/fabric_cdf_demo/silver_orders",
  "cdf_enabled": true,
  "recent_operations": [
    "DELETE",
    "OPTIMIZE",
    "UPDATE",
    "WRITE"
  ],
  "verdict": "WOULD FULL-REFRESH",
  "reasons": [
    "non-append operation(s) since v0: ['DELETE', 'UPDATE'] - incremental refresh requires append-only source data, even with CDF on"
  ]
}

-> This is exactly the case nb_cdf_incremental_pattern (this notebook) exists to handle:
   CDF + a SQL Database watermark, doing what MLV's append-only requirement excludes.


## 10 — The same pattern in pure Spark SQL
Section 20 of the internals doc shows this as a code pair; here it actually runs, against the same
`silver_orders` / `gold_daily` tables, proving the SQL (not just the Python) does what's claimed:
version-bounded `table_changes()`, net-change collapse, a scoped recompute, and a forward-only
watermark MERGE - plus the idempotency check repeated in SQL.

In [12]:
# A plain Delta control table for the watermark - self-contained, no external dependency,
# so this section runs anywhere. The Fabric SQL Database version swaps only this table for
# pyodbc reads/writes (Sec 27); the CDF logic below is unchanged either way.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS delta.`{TABLES_ROOT}/ctl_cdf_watermark`
  (entity_name STRING, last_version BIGINT, updated_at TIMESTAMP)
USING DELTA
""")
spark.sql(f"""
MERGE INTO delta.`{TABLES_ROOT}/ctl_cdf_watermark` t
USING (SELECT 'sql_demo_entity' AS entity_name, 0L AS last_version) s
ON t.entity_name = s.entity_name
WHEN NOT MATCHED THEN INSERT (entity_name, last_version, updated_at)
  VALUES (s.entity_name, s.last_version, current_timestamp())
""")
CTL = f"{TABLES_ROOT}/ctl_cdf_watermark"
print("watermark seeded at version", spark.sql(
    f"SELECT last_version FROM delta.`{CTL}` WHERE entity_name='sql_demo_entity'").collect()[0][0])

watermark seeded at version 0


In [13]:
# Read the watermark + target version (the one piece of Python glue Spark 3.5 needs -
# Spark 4.x would express this with DECLARE/SET VAR instead, no Python at all).
# Identifiers (table paths) cannot be parameter-markered - only values can (:e below).
wm_row = spark.sql(
    f"SELECT last_version FROM delta.`{CTL}` WHERE entity_name = :e",
    args={"e": "sql_demo_entity"}).collect()
sql_wm = wm_row[0][0] if wm_row else 0
sql_target = current_version(SILVER)
print(f"SQL pattern: watermark={sql_wm} target={sql_target}")

# table_changes() needs a resolvable table identifier, not an inline delta.`path` string -
# register the path once as a named table reference (this is also how Fabric Lakehouse
# tables are normally addressed - by name, not by raw path).
spark.sql(f"CREATE TABLE IF NOT EXISTS silver_orders_named USING DELTA LOCATION '{SILVER}'")

# table_changes() SQL table-valued function - version bounds pushed into the source,
# net-change collapsed via a window + outer filter (portable form; QUALIFY works identically
# on both runtimes and is shown in the doc).
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW v_net_changes AS
SELECT order_id, order_date, amount, status FROM (
  SELECT *, ROW_NUMBER() OVER (
    PARTITION BY order_id ORDER BY _commit_version DESC) AS rn
  FROM table_changes('silver_orders_named', {sql_wm + 1}, {sql_target})
  WHERE _change_type != 'update_preimage' AND status = 'complete'
) WHERE rn = 1
""")
net_count = spark.sql("SELECT count(*) FROM v_net_changes").collect()[0][0]
print(f"net change rows: {net_count}")

spark.sql("""
CREATE OR REPLACE TEMP VIEW v_recomputed_gold AS
SELECT s.order_date, SUM(s.amount) AS revenue, COUNT(*) AS orders
FROM v_net_changes n
JOIN delta.`""" + SILVER + """` s ON s.order_date = n.order_date AND s.status = 'complete'
GROUP BY s.order_date
""")
touched = spark.sql("SELECT count(*) FROM v_recomputed_gold").collect()[0][0]
print(f"gold grains recomputed: {touched}")

SQL pattern: watermark=0 target=5


net change rows: 23000


gold grains recomputed: 10


In [14]:

spark.sql(f"""
MERGE INTO delta.`{GOLD}` t
USING v_recomputed_gold s
ON t.order_date = s.order_date
WHEN MATCHED THEN UPDATE SET t.revenue = s.revenue, t.orders = s.orders
WHEN NOT MATCHED THEN INSERT (order_date, revenue, orders)
  VALUES (s.order_date, s.revenue, s.orders)
""")
spark.sql(f"""
MERGE INTO delta.`{CTL}` t
USING (SELECT 'sql_demo_entity' AS entity_name, {sql_target} AS new_version) s
ON t.entity_name = s.entity_name
WHEN MATCHED AND s.new_version > t.last_version
  THEN UPDATE SET last_version = s.new_version, updated_at = current_timestamp()
""")
new_wm = spark.sql(f"SELECT last_version FROM delta.`{CTL}` WHERE entity_name='sql_demo_entity'").collect()[0][0]
print(f"gold row count: {spark.sql(f'SELECT count(*) FROM delta.`{GOLD}`').collect()[0][0]}")
print(f"watermark advanced: {sql_wm} -> {new_wm}")
assert new_wm == sql_target


gold row count: 10
watermark advanced: 0 -> 5


In [15]:

# Re-run the SQL pattern once more with no new changes - must process zero net rows.
wm2 = spark.sql(f"SELECT last_version FROM delta.`{CTL}` WHERE entity_name='sql_demo_entity'").collect()[0][0]
tgt2 = current_version(SILVER)
print(f"second SQL run: watermark={wm2} target={tgt2}")
if wm2 >= tgt2:
    print("PASS (SQL): nothing new since last run - correctly a no-op")
else:
    n2 = spark.sql(f"""
        SELECT count(*) FROM table_changes('silver_orders_named', {wm2 + 1}, {tgt2})
        WHERE _change_type != 'update_preimage'
    """).collect()[0][0]
    assert n2 == 0, "SQL pattern idempotency broken - second run found rows to apply"
    print(f"PASS (SQL): {n2} rows found in the new version window - correctly zero")


second SQL run: watermark=5 target=5
PASS (SQL): nothing new since last run - correctly a no-op


In [16]:
def earliest_available_version(path):
    return min(h["version"] for h in spark.sql(f"DESCRIBE HISTORY delta.`{path}`").collect())

def read_changes_guarded(path, from_version, to_version, cols, predicate=None):
    earliest = earliest_available_version(path)
    if from_version + 1 < earliest:
        raise LookupError(
            f"watermark {from_version} predates earliest retained version {earliest} - "
            f"CDF history has aged out. Fall back to FULL refresh and reset the watermark.")
    return read_changes(path, from_version, to_version, cols, predicate)

# Simulate a stalled consumer whose watermark has aged out.
try:
    read_changes_guarded(SILVER, -5, latest2, ["order_id"])
    print("no guard triggered")
except LookupError as e:
    print("GUARD FIRED ->", e)
    print("recovery: full recompute of gold, then set_watermark(entity, source, current_version, n)")

print("\nOperational rule: alert on (current_version - last_version) growing, and on watermark age "
      "approaching delta.logRetentionDuration / VACUUM retention - lag is the leading indicator.")
mcon.close(); spark.stop()

GUARD FIRED -> watermark -5 predates earliest retained version 0 - CDF history has aged out. Fall back to FULL refresh and reset the watermark.
recovery: full recompute of gold, then set_watermark(entity, source, current_version, n)

Operational rule: alert on (current_version - last_version) growing, and on watermark age approaching delta.logRetentionDuration / VACUUM retention - lag is the leading indicator.
